# dlc-link-live: annotated live view

Runs `dlc-link-live --view` against a camera or relay stream, watching the declared
keypoints (with their likelihoods), the authored overlay thresholds, the pilot's run
identity, and the FDA state -- all updating while the task runs on the Pi.

**This notebook writes nothing, anywhere.** `dlc-link-live` writes no file by default
(D-47), and this notebook adds no write of its own.

**Never install `opencv-python` into this environment.** Only
`opencv-python-headless` is installed here; both packages ship the same `cv2` module,
and whichever installs second silently overwrites the other -- taking the DeepLabCut
stack down with it. The headless build has no interactive window-display function at
all, which is why this notebook (and `dlc-link-live` itself) never calls one.

Run the cells in order: parameters, then the sender+display cell, then the stop cell
once you are done watching.


In [ ]:
# Edit these values -- never the cells below. Comments explain what each one is for.

# On THIS rig the camera is GigE Vision, in the rig room, and the GPU box
# (YizharGPU12) is in a different room (D-87). The cable cannot be moved, so --source
# is the MJPEG relay URL an ffmpeg process on the LAB computer serves -- NEVER a
# device index like "0". Ask whoever ran the runbook's relay step for the exact URL;
# it looks like this:
source = "http://<lab-computer-lan-ip>:8080/"

model_path = "/path/to/exported_model.pt"  # the .pt file from deeplabcut.export_model(...)
signal_map = "/path/to/<source_id>_signals.py"  # generated by dlc-link-generate

host = "132.77.73.125"  # the pilot's orchestrator host -- ask the lab, not a guess
port = 5601  # the (pilot, source_id) row's port in pilot_hardware_config -- NOT a guess

source_id = None  # None means "use the signal map's own SOURCE_ID"
pilot_name = "RecordingBox"  # the pilot name as the orchestrator keys it

orchestrator_url = "http://132.77.73.125:9000"  # None disables run-identity polling
es_url = "http://132.77.73.217:9200"  # None disables FDA-state polling
es_index = "event_log_v2"

overlay = None  # e.g. "nose_x>0.50,nose_likelihood>0.6" -- your own authored numbers,
# never read from the task definition (D-60). None draws no threshold lines.

view_min_likelihood = 0.6  # from YOUR measured likelihood distribution (D-41,
# 35-HARDWARE-VALIDATION.md §5b) -- this tool refuses to invent one.

max_seconds = 120.0  # the notebook-friendly bounded run (D-55); None runs until the
# source ends or the kernel is interrupted.


In [ ]:
import threading
import time

import dlc_link.live_cli

argv = [
    "--source", source,
    "--model-path", model_path,
    "--signal-map", signal_map,
    "--host", host,
    "--port", str(port),
    "--view",
    "--view-min-likelihood", str(view_min_likelihood),
    "--pilot", pilot_name,
]
if source_id:
    argv += ["--source-id", source_id]
if orchestrator_url:
    argv += ["--orchestrator-url", orchestrator_url]
if es_url:
    argv += ["--es-url", es_url]
if es_index:
    argv += ["--es-index", es_index]
if overlay:
    argv += ["--overlay", overlay]
if max_seconds:
    argv += ["--max-seconds", str(max_seconds)]

# dlc_link.live_cli.main writes NOTHING anywhere (D-47) and never opens an
# interactive display window -- opencv-python-headless has no such function, and
# opencv-python must never be installed beside it (it would clobber this
# environment's own DLC build).
_handles = {"viewer": None, "sink": None, "exit_code": None}


def _on_viewer_ready(viewer, sink):
    _handles["viewer"] = viewer
    _handles["sink"] = sink


def _run_sender():
    _handles["exit_code"] = dlc_link.live_cli.main(argv, on_viewer_ready=_on_viewer_ready)


sender_thread = threading.Thread(target=_run_sender, daemon=True)
sender_thread.start()

while _handles["sink"] is None and sender_thread.is_alive():
    time.sleep(0.05)

# The display loop runs in THIS cell's own execution thread, on purpose: IPython's
# display()/clear_output() are meant to be driven from the kernel's own thread, never
# from dlc_link.live_cli.main's background thread. Interrupting the kernel raises
# KeyboardInterrupt HERE, which stops this loop cooperatively -- it does not stop the
# sender thread, whose own `finally` closes the mics-link connection and the capture
# device on ITS OWN termination (--max-seconds above, the source ending, or a kernel
# restart).
try:
    while sender_thread.is_alive():
        if _handles["sink"] is not None:
            _handles["sink"].update()
        time.sleep(0.2)
except KeyboardInterrupt:
    print("Display loop interrupted. The sender thread is still running in the "
          "background -- run the next cell to stop the viewer and see the final "
          "summary.")


In [ ]:
# Stops the viewer's own threads (rendering/polling) and prints the final status.
# The sender thread itself stops on its own termination (above); this cell does not
# force it to stop early -- interrupting the kernel, or waiting out --max-seconds, does
# that.
if _handles["viewer"] is not None:
    _handles["viewer"].stop()
    print("\n".join(_handles["viewer"].status.lines()))
else:
    print("The viewer was never constructed -- the sender may have already finished, "
          "or --view did not reach that point. Check the output above this cell.")

sender_thread.join(timeout=5.0)
print("sender thread finished:", not sender_thread.is_alive())
print("sender exit code:", _handles["exit_code"])
